## Trying the interactive quickstart with the full dataset 

looking at the full dataset to see how feasible it is to find the cell cycle in t-SNE dimension reduction

In [1]:
from pathlib import Path
import sys

import torch
import pandas as pd


A module that was compiled using NumPy 1.x cannot be run in
NumPy 2.4.2 as it may crash. To support both 1.x and 2.x
versions of NumPy, modules must be compiled with NumPy 2.0.
Some module may need to rebuild instead e.g. with 'pybind11>=2.12'.

If you are a user of the module, the easiest solution will be to
downgrade to 'numpy<2' or try to upgrade the affected module.
We expect that some modules will need time to support NumPy 2.

Traceback (most recent call last):  File "<frozen runpy>", line 198, in _run_module_as_main
  File "<frozen runpy>", line 88, in _run_code
  File "/Users/alexiahohn/miniforge3/envs/phenome-gpu/lib/python3.12/site-packages/ipykernel_launcher.py", line 18, in <module>
    app.launch_new_instance()
  File "/Users/alexiahohn/miniforge3/envs/phenome-gpu/lib/python3.12/site-packages/traitlets/config/application.py", line 1080, in launch_instance
    app.start()
  File "/Users/alexiahohn/miniforge3/envs/phenome-gpu/lib/python3.12/site-packages/ipykernel/kernelapp

In [2]:
# Local repository / Windows setup
from windows_setup import setup_local_phenome

REPO_ROOT = setup_local_phenome(verify_imports=True)
print("Repository root:", REPO_ROOT)
print("✓ Local environment configured.")

Repository root: /Users/alexiahohn/info/MA1/ML/ML_project2/ML_PhenoMe
✓ Local environment configured.


In [3]:
# Import project and dataset helpers from the companion modules
from phenome import PhenoMe, load_dinov2_model, make_dataframe_metadata_fn
from fucci_dataset_utils import (
    FUCCI_SUBSET_CONFIGS,
    prepare_fucci_subset,
    resolve_fucci_roots,
)

In [4]:
import torch

if torch.backends.mps.is_available():
    device = torch.device("mps")
else:
    device = torch.device("cpu")

seed = 42

raw_root, out_root = resolve_fucci_roots()

print("Device:", device)



Device: mps


In [5]:
images_dir, meta_path = prepare_fucci_subset(
    "train",       
    raw_root=raw_root,
    out_root=out_root,
    max_tracks=None,  # all tracks
    frame_stride=1,   # every frame
)

✓ train: exported 1195196 frames from 5188 tracks -> /Users/alexiahohn/info/MA1/ML/ML_project2/ML_PhenoMe/fucci_toy_datasets/train/FUCCI_images


In [6]:

metadata_df = pd.read_csv(meta_path)

file_df = pd.DataFrame({
    "file_path": [
        str(Path(images_dir) / filename)
        for filename in metadata_df["filename"]
    ]
})

file_df = pd.concat(
    [file_df, metadata_df.drop(columns=["filename"])],
    axis=1,
)

pheno = PhenoMe(device=device, seed=seed)
file_df = pheno.set_file_df(
    file_df,
    channel_names=["Brightfield", "H2B"],
)

print("Loaded rows:", len(file_df))
display(file_df.head())
display(pheno.inspect_data())

Loaded rows: 1195196


,file_path,source_file,track_id,frame,num_frames,time_min,label_len,dataset_split,condition
0,/Users/alexiahohn/info/MA1/ML/ML_project2/ML_P...,0403.1.npy,403.1,0,338,0,338,train,train
1,/Users/alexiahohn/info/MA1/ML/ML_project2/ML_P...,0403.1.npy,403.1,1,338,5,338,train,train
2,/Users/alexiahohn/info/MA1/ML/ML_project2/ML_P...,0403.1.npy,403.1,2,338,10,338,train,train
3,/Users/alexiahohn/info/MA1/ML/ML_project2/ML_P...,0403.1.npy,403.1,3,338,15,338,train,train
4,/Users/alexiahohn/info/MA1/ML/ML_project2/ML_P...,0403.1.npy,403.1,4,338,20,338,train,train


Inspecting 1195196 images...


Reading data:   0%|          | 0/1195196 [00:00<?, ?it/s]

Unique shapes: 1  |  H: 64-64  |  W: 64-64  |  C: 2-2
Top 5 shapes:
  1. (64, 64, 2)  →  1195196 images
Biggest:   (64, 64, 2)  [8192 px]
Smallest:  (64, 64, 2)  [8192 px]
Top 3 dtypes:
  1. float16  →  1195196 images
Top 3 data ranges [min, max]:
  1. [-5.95, 4.01]  →  12 images
  2. [-6.57, 3.95]  →  12 images
  3. [-6.39, 4.27]  →  11 images
Global intensity range: min=-19.48  max=26.20


,file_path,height,width,channels,dtype,min,max,error,shape_match,mask_height,mask_width,mask_dtype,mask_min,mask_max,mask_error
0,/Users/alexiahohn/info/MA1/ML/ML_project2/ML_P...,64,64,2,float16,-8.046875,4.351562,None,None,NaN,NaN,None,NaN,NaN,None
1,/Users/alexiahohn/info/MA1/ML/ML_project2/ML_P...,64,64,2,float16,-7.035156,4.171875,None,None,NaN,NaN,None,NaN,NaN,None
2,/Users/alexiahohn/info/MA1/ML/ML_project2/ML_P...,64,64,2,float16,-7.875000,4.589844,None,None,NaN,NaN,None,NaN,NaN,None
3,/Users/alexiahohn/info/MA1/ML/ML_project2/ML_P...,64,64,2,float16,-8.601562,4.730469,None,None,NaN,NaN,None,NaN,NaN,None
4,/Users/alexiahohn/info/MA1/ML/ML_project2/ML_P...,64,64,2,float16,-6.972656,4.910156,None,None,NaN,NaN,None,NaN,NaN,None
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1195191,/Users/alexiahohn/info/MA1/ML/ML_project2/ML_P...,64,64,2,float16,-5.226562,4.945312,None,None,NaN,NaN,None,NaN,NaN,None
1195192,/Users/alexiahohn/info/MA1/ML/ML_project2/ML_P...,64,64,2,float16,-5.117188,6.859375,None,None,NaN,NaN,None,NaN,NaN,None
1195193,/Users/alexiahohn/info/MA1/ML/ML_project2/ML_P...,64,64,2,float16,-5.855469,7.671875,None,None,NaN,NaN,None,NaN,NaN,None
1195194,/Users/alexiahohn/info/MA1/ML/ML_project2/ML_P...,64,64,2,float16,-5.558594,7.277344,None,None,NaN,NaN,None,NaN,NaN,None


In [ ]:
model_wrapper = load_dinov2_model(
    model_name="dinov2_vits14_reg",
    device=device,
)

print("Wrapper device:", model_wrapper.device)
print("Model device:", next(model_wrapper.model.parameters()).device)

pheno.process_images(
    model_wrapper,
    num_workers=0,
    batch_size=4,
    channel_mode="split",
    resize_size=224,
    checkpoint_path="fucci_full_train_embeddings.h5",
)

Using cache found in /Users/alexiahohn/.cache/torch/hub/facebookresearch_dinov2_main
/Users/alexiahohn/.cache/torch/hub/facebookresearch_dinov2_main/dinov2/layers/swiglu_ffn.py:51: UserWarning:

xFormers is not available (SwiGLU)

/Users/alexiahohn/.cache/torch/hub/facebookresearch_dinov2_main/dinov2/layers/attention.py:33: UserWarning:

xFormers is not available (Attention)

/Users/alexiahohn/.cache/torch/hub/facebookresearch_dinov2_main/dinov2/layers/block.py:40: UserWarning:

xFormers is not available (Block)



Wrapper device: mps
Model device: mps:0


In [ ]:
figure = pheno.plot_tsne(
    n_components=2,
    perplexity=30,
    color_by="time_min",
    sample_size=10_000,
    return_fig=True,
)

display(figure)